In [ ]:
operation = "blocked"
probe_id = ""
run_config_json = "{}"
selected_demos_json = "[]"


In [ ]:
import base64
import json
import re
import uuid
from datetime import datetime, timezone
spark.conf.set("spark.fabric.session.highConcurrency.enabled", "true")
config = json.loads(run_config_json)
selected = json.loads(selected_demos_json)
if operation == "preflight":
    if not isinstance(probe_id, str) or not re.fullmatch(r"[a-z0-9][a-z0-9-]{0,63}", probe_id):
        raise ValueError("A safe, unique preflight probe_id is required")
    context = notebookutils.runtime.context
    def token_identity(audience):
        token = notebookutils.credentials.getToken(audience)
        payload = token.split(".")[1]
        claims = json.loads(base64.urlsafe_b64decode(payload + "=" * (-len(payload) % 4)))
        if not claims.get("oid") or not claims.get("tid"):
            raise RuntimeError("Token lacks identity claims")
        return claims["oid"], claims["tid"]
    storage_oid, storage_tenant = token_identity("storage")
    evidence = {"probe_id": probe_id, "operation": operation, "parameter_marker": config.get("probe_marker"), "selected_demos": selected, "is_for_pipeline": context["isForPipeline"], "workspace_id": context["currentWorkspaceId"], "lakehouse_id": context["defaultLakehouseId"], "storage_principal_oid": storage_oid, "tenant_id": storage_tenant}
    failures = []
    if config.get("probe_kusto"):
        try:
            kusto_oid, kusto_tenant = token_identity("kusto")
            evidence.update(kusto_principal_oid=kusto_oid, kusto_tenant_id=kusto_tenant)
        except Exception as exc:
            evidence["kusto_token_error"] = type(exc).__name__
            failures.append("Kusto token acquisition")
        try:
            import azure.identity
            import azure.kusto.data
            import azure.kusto.ingest
            evidence["kusto_sdk_available"] = True
        except ImportError as exc:
            evidence["kusto_sdk_available"] = False
            evidence["kusto_sdk_error"] = type(exc).__name__
            failures.append("Kusto SDK imports")
    if config.get("probe_bundle"):
        try:
            import fabric_delivery
            import fabric_ledger
            import fabric_runner
            import fabric_runtime
            import fabric_storage
            if "demo-2" in selected:
                import eventhouse_core
                import ingestion
                if not callable(getattr(ingestion.EventhouseAdapter, "ingestion_status", None)):
                    raise ImportError("Eventhouse terminal status adapter is missing")
                evidence["eventhouse_adapter_available"] = True
            evidence["producer_bundle_available"] = True
        except ImportError as exc:
            evidence["producer_bundle_available"] = False
            evidence["producer_bundle_error"] = type(exc).__name__
            failures.append("Producer module imports")
    if config.get("probe_storage"):
        result = {"path": "Files/preflight/storage-" + probe_id + ".txt", "deleted": False, "step": "create"}
        try:
            from fabric_storage import OneLakeStore, StorageConflict
            store = OneLakeStore(context["currentWorkspaceId"], context["defaultLakehouseId"], lambda: notebookutils.credentials.getToken("storage"))
            first_etag = store.create(result["path"], b"before")
            if store.read(result["path"]) != (b"before", first_etag):
                raise RuntimeError("Conditional create readback differed")
            try:
                store.create(result["path"], b"duplicate")
            except StorageConflict:
                result["duplicate_create_rejected"] = True
            else:
                raise RuntimeError("Duplicate create was not rejected")
            try:
                store.replace(result["path"], b"stale", '"stale-etag"')
            except StorageConflict:
                result["stale_update_rejected"] = True
            else:
                raise RuntimeError("Stale update was not rejected")
            new_etag = store.replace(result["path"], b"after", first_etag)
            if store.read(result["path"]) != (b"after", new_etag):
                raise RuntimeError("Conditional update readback differed")
            result["update_readback_verified"] = True
            result["step"] = "stale-delete"
            try:
                store.delete(result["path"], first_etag)
            except StorageConflict:
                result["stale_delete_rejected"] = True
            else:
                raise RuntimeError("Stale delete was not rejected")
            if store.read(result["path"]) != (b"after", new_etag):
                raise RuntimeError("Stale delete changed the storage sentinel")
            result["step"] = "delete"
            store.delete(result["path"], new_etag)
            result["step"] = "confirm-absent"
            try:
                store.read(result["path"])
            except FileNotFoundError:
                result["deleted"] = True
            else:
                raise RuntimeError("Storage sentinel remains")
        except Exception as exc:
            result["error"] = type(exc).__name__
            if type(exc).__name__ in ("StorageError", "StorageConflict", "StorageOutcomeUnknown"):
                result["diagnostic"] = str(exc)
            failures.append("Conditional storage probe")
        evidence["conditional_storage_probe"] = result
    auth = config.get("eventhouse_auth_probe")
    if auth is not None:
        try:
            if not isinstance(auth, dict) or set(auth) != {"query_endpoint", "database"}:
                raise ValueError("Eventhouse auth probe requires endpoint and database")
            from azure.kusto.data import KustoClient, KustoConnectionStringBuilder
            from fabric_runtime import _kusto_endpoint
            endpoint = _kusto_endpoint(auth["query_endpoint"])
            database = auth["database"]
            if not isinstance(database, str) or not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]{0,127}", database):
                raise ValueError("Invalid KQL database name")
            connection = KustoConnectionStringBuilder.with_token_provider(endpoint, lambda: notebookutils.credentials.getToken("kusto"))
            response = KustoClient(connection).execute_query(database, "print ProducerRuntimeProbe=1")
            rows = list(response.primary_results[0])
            if len(rows) != 1 or rows[0]["ProducerRuntimeProbe"] != 1:
                raise RuntimeError("Eventhouse query evidence differed")
            evidence["eventhouse_sdk_query_verified"] = True
        except Exception as exc:
            evidence["eventhouse_sdk_query_verified"] = False
            evidence["eventhouse_sdk_query_error"] = type(exc).__name__
            failures.append("Eventhouse SDK query")
    access = config.get("destination_access_probe")
    if access is not None:
        if not isinstance(access, dict) or set(access) != {"workspace_id", "lakehouse_id"}:
            raise ValueError("Destination access probe requires workspace and Lakehouse IDs")
        workspace_id = str(uuid.UUID(access["workspace_id"]))
        lakehouse_id = str(uuid.UUID(access["lakehouse_id"]))
        probe_dir = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Files/_preflight/producer-access/{probe_id}"
        probe_file = probe_dir + "/identity-probe.txt"
        expected = "producer access probe " + probe_id + "\n"
        result = {"path": probe_file, "started_at_utc": datetime.now(timezone.utc).isoformat(), "readback_verified": False, "sentinel_deleted": False, "directory_deleted": False}
        created_directory = False
        try:
            if notebookutils.fs.exists(probe_dir):
                raise RuntimeError("Destination probe directory already exists")
            if not notebookutils.fs.mkdirs(probe_dir):
                raise RuntimeError("Destination probe directory creation returned false")
            created_directory = True
            if not notebookutils.fs.put(probe_file, expected, False):
                raise RuntimeError("Destination probe write returned false")
            if notebookutils.fs.head(probe_file, len(expected.encode("utf-8")) + 1) != expected:
                raise RuntimeError("Destination probe readback differed")
            result["readback_verified"] = True
        except Exception as exc:
            result["error"] = type(exc).__name__
        finally:
            if created_directory:
                try:
                    if notebookutils.fs.exists(probe_file) and not notebookutils.fs.rm(probe_file, False):
                        raise RuntimeError("Destination probe file cleanup returned false")
                    result["sentinel_deleted"] = not notebookutils.fs.exists(probe_file)
                    if not result["sentinel_deleted"] or not notebookutils.fs.rm(probe_dir, False):
                        raise RuntimeError("Destination probe directory cleanup failed")
                    result["directory_deleted"] = not notebookutils.fs.exists(probe_dir)
                    if not result["directory_deleted"]:
                        raise RuntimeError("Destination probe directory remains")
                except Exception as exc:
                    result["cleanup_error"] = type(exc).__name__
            result["completed_at_utc"] = datetime.now(timezone.utc).isoformat()
        evidence["destination_access_probe"] = result
        if not all(result[key] for key in ("readback_verified", "sentinel_deleted", "directory_deleted")):
            failures.append("Destination access probe")
    directory = "Files/preflight"
    path = directory + "/" + probe_id + ".json"
    if not notebookutils.fs.exists(directory) and not notebookutils.fs.mkdirs(directory):
        raise RuntimeError("Could not create preflight directory")
    if notebookutils.fs.exists(path):
        raise RuntimeError("Preflight evidence already exists; choose a new probe ID")
    serialized = json.dumps(evidence, sort_keys=True)
    if not notebookutils.fs.put(path, serialized, False):
        raise RuntimeError("Could not write preflight evidence")
    if notebookutils.fs.head(path, len(serialized.encode("utf-8")) + 1) != serialized:
        raise RuntimeError("Preflight evidence did not read back")
    print("Preflight evidence persisted:", path)
    if failures:
        raise RuntimeError("Preflight failed: " + ", ".join(failures))
elif operation == "run-data":
    from generator import RunConfig
    if not isinstance(selected, list) or not selected or len(selected) != len(set(selected)) or any(x not in ("demo-1", "demo-2", "demo-3") for x in selected):
        raise ValueError("Select distinct demo destinations")
    run = RunConfig(run_id=config["run_id"], seed=config["seed"], start_date=config["start_date"], date_count=config["date_count"], products=config["product_count"], customers=config["customer_count"], rows_per_file=config["rows_per_file"], file_count=config["file_count"], interval_seconds=config["interval_seconds"])
    run.validate()
    raise RuntimeError("Producer execution blocked: destination adapters and OneLake staging semantics are not verified")
else:
    raise RuntimeError("Explicit supported operation required")
